[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/misda/blob/main/examples/diagnostic_clean.ipynb)

# MISDA — clean controlled diagnostics

This notebook evaluates the unified controlled diagnostic suite under exact observation: `Y = Z = F(X)`. Ground truth is generated from the theoretical problem and the sampled clean objective matrix `Z`, never from the observed result or from MISDA.

In [ ]:
from pathlib import Path
import subprocess
import sys

# In a repository checkout, test the local code. In Colab, install main.
target = ".[benchmarks]" if Path("pyproject.toml").exists() else "git+https://github.com/monacofj/misda.git@main#egg=misda[benchmarks]"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", target])


In [ ]:
import misda
import misda.benchmarks as bench

N = 300
SEED = 123
SIGMA = 0.0

## Unified diagnostic suite

Each problem samples `X`, evaluates the clean theoretical objectives `Z=F(X)`, and uses identity observation (`sigma=0`), so MISDA receives exactly `Y=Z`.

In [ ]:
diagnostic_results = {}
for problem in bench.PROBLEMS:
    dataset = problem.generate(N=N, seed=SEED, sigma=SIGMA)
    truth = bench.diagnostic_truth(problem, dataset.Z)
    mis_set = misda.discover(dataset.Y, name=truth["name"], seed=SEED)
    misda.evaluate(mis_set, metrics=("linear", "pareto"))
    benchmark_result = misda.benchmark(mis_set, truth)
    print(benchmark_result.report())
    mis_set.graph_plot()
    diagnostic_results[problem.id] = {
        "dataset": dataset,
        "result_obj": mis_set,
        "benchmark_obj": benchmark_result,
        "truth": truth,
    }